# Checkpoint test for re-trained unsupervised SimCSE
COMP8240 — Trilok Kumar Reddy Ande

Our full re-training (`train_unsup.ipynb`) scored 72.55 against the paper's 76.25. One possible cause is that the paper evaluates on the **STS-B development set every 125 steps and keeps the best checkpoint**, while we kept the final model.

This notebook tests that cause cheaply: it trains on **100k sentences** (same settings, seed 42), scores every checkpoint on the STS-B **dev** set every 125 steps, and then evaluates both the **best-dev** and the **final** checkpoint on the STS **test** sets with the official script.

Run on a **T4 GPU**: Runtime → Run all (about 10–15 minutes).

In [1]:
%cd /content
!rm -rf SimCSE
!git clone -q https://github.com/princeton-nlp/SimCSE.git
%cd /content/SimCSE
!pip install -q transformers scipy scikit-learn prettytable
!sed -i "s/tokenizer.batch_encode_plus(/tokenizer(/g" evaluation.py
%cd SentEval/data/downstream/
!bash download_dataset.sh > /dev/null 2>&1
%cd /content
!wget -q -O /content/wiki1m.txt https://huggingface.co/datasets/princeton-nlp/datasets-for-simcse/resolve/main/wiki1m_for_simcse.txt
!wc -l /content/wiki1m.txt

/content
/content/SimCSE
/content/SimCSE/SentEval/data/downstream
/content
1000000 /content/wiki1m.txt


In [2]:
import torch, random, time, transformers
import torch.nn as nn, torch.nn.functional as F
from scipy.stats import spearmanr
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
print("torch", torch.__version__, "| transformers", transformers.__version__, "| GPU", torch.cuda.get_device_name(0))

N_TRAIN, EVAL_EVERY = 100_000, 125
BATCH, LR, MAXLEN, TEMP = 64, 3e-5, 32, 0.05
random.seed(42); torch.manual_seed(42)

lines = [l.strip() for l in open("/content/wiki1m.txt", encoding="utf-8") if l.strip()]
random.shuffle(lines)
train = lines[:N_TRAIN]

# STS-B development set (from SentEval's download)
dev_s1, dev_s2, dev_y = [], [], []
for row in open("/content/SimCSE/SentEval/data/downstream/STS/STSBenchmark/sts-dev.csv", encoding="utf-8"):
    p = row.rstrip("\n").split("\t")
    dev_y.append(float(p[4])); dev_s1.append(p[5]); dev_s2.append(p[6])
print("Training sentences:", len(train), "| STS-B dev pairs:", len(dev_y))

tok = AutoTokenizer.from_pretrained("bert-base-uncased")
enc = AutoModel.from_pretrained("bert-base-uncased").cuda().train()
mlp = nn.Sequential(nn.Linear(768, 768), nn.Tanh()).cuda()

@torch.no_grad()
def embed(sents):
    out = []
    for i in range(0, len(sents), 128):
        x = tok(sents[i:i+128], padding=True, truncation=True, max_length=MAXLEN, return_tensors="pt").to("cuda")
        out.append(enc(**x).last_hidden_state[:, 0].float())   # [CLS] without MLP, as at test time
    return torch.cat(out)

def dev_score():
    enc.eval()
    sims = F.cosine_similarity(embed(dev_s1), embed(dev_s2)).cpu().numpy()
    enc.train()
    return 100 * spearmanr(sims, dev_y).correlation

torch 2.14.1+cu130 | transformers 5.22.1 | GPU Tesla T4
Training sentences: 100000 | STS-B dev pairs: 1500


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'TypeError: Failed to fetch'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
opt = torch.optim.AdamW(list(enc.parameters()) + list(mlp.parameters()), lr=LR)
steps = len(train) // BATCH
sched = get_linear_schedule_with_warmup(opt, 0, steps)
scaler = torch.amp.GradScaler()

log = [(0, dev_score())]
print(f"step     0  dev {log[-1][1]:.2f}")
best = log[0][1]
t0 = time.time()
for step in range(1, steps + 1):
    x = tok(train[(step-1) * BATCH:step * BATCH], padding=True, truncation=True,
            max_length=MAXLEN, return_tensors="pt").to("cuda")
    with torch.autocast("cuda", dtype=torch.float16):
        z1 = mlp(enc(**x).last_hidden_state[:, 0])
        z2 = mlp(enc(**x).last_hidden_state[:, 0])
    sim = F.cosine_similarity(z1.float().unsqueeze(1), z2.float().unsqueeze(0), dim=-1) / TEMP
    loss = F.cross_entropy(sim, torch.arange(sim.size(0), device="cuda"))
    opt.zero_grad()
    scaler.scale(loss).backward()
    scaler.step(opt); scaler.update(); sched.step()
    if step % EVAL_EVERY == 0 or step == steps:
        d = dev_score(); log.append((step, d))
        flag = ""
        if d > best:
            best = d; flag = "  <- best so far (saved)"
            enc.save_pretrained("/content/ckpt-best"); tok.save_pretrained("/content/ckpt-best")
        print(f"step {step:5d}  loss {loss.item():.4f}  dev {d:.2f}  {time.time()-t0:.0f}s{flag}")

enc.save_pretrained("/content/ckpt-final"); tok.save_pretrained("/content/ckpt-final")
scores = [d for s, d in log if s > 0]
print()
print(f"Dev scores after training starts: min {min(scores):.2f}, max {max(scores):.2f}, range {max(scores)-min(scores):.2f}")
print(f"Best step {max(log[1:], key=lambda t: t[1])[0]} (dev {max(scores):.2f}) | final step {steps} (dev {scores[-1]:.2f})")
print("Training finished in", round((time.time() - t0) / 60, 1), "minutes")

step     0  dev 30.18


/tmp/ipykernel_3512/3616765845.py:20: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


step   125  loss 0.0005  dev 53.74  43s  <- best so far (saved)


step   250  loss 0.0009  dev 56.14  75s  <- best so far (saved)


step   375  loss 0.0002  dev 60.26  110s  <- best so far (saved)


step   500  loss 0.0012  dev 64.20  147s  <- best so far (saved)


... (6 outputs left out)

step  1375  loss 0.0002  dev 68.25  476s  <- best so far (saved)


step  1500  loss 0.0001  dev 68.27  544s  <- best so far (saved)


step  1562  loss 0.0001  dev 68.29  568s  <- best so far (saved)



Dev scores after training starts: min 53.74, max 68.29, range 14.56
Best step 1562 (dev 68.29) | final step 1562 (dev 68.29)
Training finished in 9.6 minutes


## Test-set scores: best-dev checkpoint vs final checkpoint

In [4]:
%cd /content/SimCSE
print("=== BEST-DEV checkpoint ===")
!python evaluation.py --model_name_or_path /content/ckpt-best --pooler cls_before_pooler --task_set sts --mode test 2>&1 | grep -A 4 "STS12" | head -n 5
print("=== FINAL checkpoint ===")
!python evaluation.py --model_name_or_path /content/ckpt-final --pooler cls_before_pooler --task_set sts --mode test 2>&1 | grep -A 4 "STS12" | head -n 5

/content/SimCSE
=== BEST-DEV checkpoint ===
2026-11-07 05:11:31,949 : ***** Transfer task : STS12 *****


2026-11-07 05:11:35,677 : MSRpar : pearson = 0.3593, spearman = 0.4052
2026-11-07 05:11:37,075 : MSRvid : pearson = 0.6280, spearman = 0.6402
=== FINAL checkpoint ===
2026-11-07 05:13:19,662 : ***** Transfer task : STS12 *****


2026-11-07 05:13:23,212 : MSRpar : pearson = 0.3593, spearman = 0.4052
2026-11-07 05:13:24,568 : MSRvid : pearson = 0.6280, spearman = 0.6402


**Result.** The STS-B dev score rose at every check (30.18 for plain BERT, 53.74 at step 125, 67.42 at step 1,000, 68.29 at the final step 1,562), so the best-dev checkpoint *is* the final checkpoint and the two test evaluations above are of the same model. (The `grep` in the last cell only caught the first STS12 log lines, not the summary table; since the models are identical this does not affect the conclusion.) At this scale, keeping the best dev checkpoint would not have changed the result.